In [ ]:
from typing import TypedDict, Annotated, List
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END, add_messages
from langgraph.checkpoint.memory import MemorySaver

# 1. 定义状态
class State(TypedDict):
    messages: Annotated[List, add_messages]

# 2. 初始化模型
model = ChatOpenAI(
    model="qwen3-max",
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    api_key="sk-8b293e101d5e41f1b3116d9848b7ac59"
)
# 3. 定义节点
def chatbot(state: State) -> dict:
    response = model.invoke(state["messages"])
    return {"messages": [response]}

# 4. 构建图
graph = StateGraph(State)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)

# 5. 编译（带记忆）
memory = MemorySaver()
app = graph.compile(checkpointer=memory)

# 6. 使用
config = {"configurable": {"thread_id": "conversation-1"}}

# 第一轮
result = app.invoke({"messages": [HumanMessage(content="你好！")]}, config)
print(result["messages"][-1].content)

# 第二轮（记住上下文）
result = app.invoke({"messages": [HumanMessage(content="我叫小明，你呢？")]}, config)
print(result["messages"][-1].content)